In [ ]:
# Example 1: Public API surface inventory.
import biotraj

public_names = [n for n in dir(biotraj) if not n.startswith('_')]
print(f"biotraj has {len(public_names)} public attributes")
print("Sample of public names:")
for n in public_names[:30]:
    obj = getattr(biotraj, n)
    print(f"  {n}: {type(obj).__name__}")

assert len(public_names) > 0, "biotraj should expose a public API"


In [ ]:
# Example 2: One file class per trajectory format
# biotraj reads and writes trajectories through these classes, each defined in its format module.
import biotraj

formats = {
    "xtc": "XTCTrajectoryFile",
    "trr": "TRRTrajectoryFile",
    "dcd": "DCDTrajectoryFile",
    "netcdf": "NetCDFTrajectoryFile",
}
for module_name, class_name in formats.items():
    cls = getattr(biotraj, class_name)
    assert getattr(getattr(biotraj, module_name), class_name) is cls, class_name
    for method in ("read", "write", "seek", "tell", "close"):
        assert callable(getattr(cls, method)), f"{class_name}.{method}"
    print(f"  biotraj.{module_name}: {class_name}")

In [ ]:
# Example 3: TrajectoryFile classes — discover and inspect.
import inspect

import biotraj

trajfile_classes = []
for name in dir(biotraj):
    obj = getattr(biotraj, name)
    if inspect.isclass(obj) and 'TrajectoryFile' in name:
        trajfile_classes.append(name)
        print(f"  {name}: methods = {[m for m in dir(obj) if not m.startswith('_')][:8]}")

print(f"\nFound {len(trajfile_classes)} *TrajectoryFile classes")


In [ ]:
# Example 4: Write and read back an XTC trajectory
import os
import tempfile

import numpy as np

import biotraj

n_frames, n_atoms = 5, 10
xyz = np.random.RandomState(42).rand(n_frames, n_atoms, 3).astype(np.float32)
time = np.arange(n_frames, dtype=np.float32) * 2.0
step = np.arange(n_frames, dtype=np.int32)

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "test.xtc")
    with biotraj.XTCTrajectoryFile(path, "w") as f:
        f.write(xyz, time=time, step=step)
    with biotraj.XTCTrajectoryFile(path, "r") as f:
        assert len(f) == n_frames
        xyz_read, time_read, step_read, _box = f.read()

print(f"wrote/read XTC: in={xyz.shape}, out={xyz_read.shape}")
assert xyz_read.shape == xyz.shape
# XTC stores coordinates to a precision of 1/1000, so compare within that tolerance
assert np.allclose(xyz_read, xyz, atol=1e-3), np.abs(xyz_read - xyz).max()
assert np.array_equal(time_read, time) and np.array_equal(step_read, step)

In [ ]:
# Example 5: Opening a missing file raises OSError
import os
import tempfile

import biotraj

with tempfile.TemporaryDirectory() as tmp:
    missing = os.path.join(tmp, "missing.xtc")
    try:
        biotraj.XTCTrajectoryFile(missing, "r")
    except OSError as err:
        print(f"caught expected error: {type(err).__name__}")
    else:
        raise AssertionError("opening a missing XTC file did not raise OSError")